Сравню две версии одной модели для распознавания сущностей. Обе обучу на одинаковых предложениях из CoNLL-2003. В одном случае использую метки, сгенерированные LLM, в другом - метки из датасета. Так проверю, как источник разметки влияет на качество

In [2]:
%pip install -q datasets

from datasets import load_dataset

dataset = load_dataset("lhoestq/conll2003")
train = dataset["train"]
validation = dataset["validation"]
test = dataset["test"]

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


Загрузил CoNLL-2003. Из train возьму предложения для обучения, а validation и test использую для оценки

In [3]:
%pip install -q openai

import json
from pathlib import Path

from google.colab import drive, userdata
from openai import OpenAI

drive.mount("/content/drive")

runs_dir = Path("/content/drive/MyDrive/llm-ner-conll2003/runs")
runs_dir.mkdir(parents=True, exist_ok=True)

client = OpenAI(
    base_url="https://litellm.ispras.ru/v1",
    api_key=userdata.get("LITELLM_API_KEY"),
)

Mounted at /content/drive


Подключил LiteLLM, чтобы получить метки для train, и Google Drive, чтобы сохранить их для обучения маленькой модели

In [6]:
tag_names = [
    "O", "B-PER", "I-PER", "B-ORG", "I-ORG",
    "B-LOC", "I-LOC", "B-MISC", "I-MISC",
]

Взял порядок меток из описания CoNLL-2003. Числовой код метки используется как индекс в этом списке

Перенесу функции из 1conll.ipynb, чтобы размечать train тем же промптом и с той же проверкой ответов, которые использовал для итогового прогона на первом этапе

In [13]:
def make_prompt(tokens):
    return f"""
Label the named entities in this English text using BIO tags.

Entity types:
- PER: person
- ORG: organization
- LOC: location
- MISC: other named entities, including nationality adjectives

Use B-TYPE for the first token of an entity, I-TYPE for its continuation,
and O for tokens outside any entity.

Return only a JSON array of strings, with exactly one tag per input token.
Keep the original token order.

Tokens: {json.dumps(tokens)}
""".strip()

make_prompt - принимает список токенов и собирает из инструкции и токенов одну строку для запроса к GPT. Токены записываю в JSON, чтобы явно передать их границы и порядок. Прошу модель вернуть по одной BIO-метке на токен

In [8]:
def parse_llm_tags(answer, token_count, allowed_tags):
    tags = json.loads(answer)

    if not isinstance(tags, list):
        raise ValueError("Ответ модели должен быть JSON-массивом меток")
    if len(tags) != token_count:
        raise ValueError("Число меток не совпадает с числом токенов")

    previous_tag = "O"

    for tag in tags:
        if tag not in allowed_tags:
            raise ValueError(f"Неизвестная метка: {tag}")

        if tag.startswith("I-"):
            begin_tag = f"B-{tag[2:]}"
            if previous_tag not in (begin_tag, tag):
                raise ValueError(f"Метка {tag} стоит после {previous_tag}")

        previous_tag = tag

    return tags

parse_llm_tags - читает строку ответа как JSON и возвращает список меток, если он прошёл проверку. Проверяю, что меток столько же, сколько токенов, что каждая метка есть в tag_names и что продолжение сущности стоит после начала или продолжения того же типа. Например, I-LOC после O считаю ошибкой. Если модель поставила LOC вместо ORG, ответ может пройти эту проверку, а ошибку в типе покажет сравнение с эталоном

In [11]:
def parse_model_tags(answer, token_count, allowed_tags):
    if not isinstance(answer, str):
        raise ValueError("Модель не вернула строку")

    lines = answer.strip().splitlines()
    if (
        len(lines) >= 2
        and lines[0].strip().lower() in ("```json", "```")
        and lines[-1].strip() == "```"
    ):
        answer = "\n".join(lines[1:-1])

    return parse_llm_tags(answer, token_count, allowed_tags)

parse_model_tags - подготавливает ответ перед проверкой BIO. На первом этапе Gemma иногда присылала массив внутри блока кода Markdown, и json.loads не мог прочитать его целиком. Если в ответе есть открывающая и закрывающая строки такого блока, убираю их и передаю содержимое в parse_llm_tags

In [12]:
def save_answer(run_path, result):
    with run_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

save_answer - записывает результат одного запроса отдельной строкой в файл JSONL. Открываю файл в режиме добавления, поэтому каждый новый ответ дописывается в конец. Сохраняю номер записи и исходный ответ, чтобы потом найти нужное предложение в train, разобрать ошибки и продолжить разметку после перезапуска Colab

In [14]:
pilot_path = runs_dir / "train_pilot_01.jsonl"
teacher_model = "openai/gpt-oss-120b"
saved_indices = set()

if pilot_path.exists():
    with pilot_path.open(encoding="utf-8") as file:
        for line in file:
            saved_indices.add(json.loads(line)["row_index"])

for row_index in range(5):
    if row_index in saved_indices:
        print(f"Запись {row_index}: ответ уже сохранён")
        continue

    row_tokens = train[row_index]["tokens"]
    response = client.chat.completions.create(
        model=teacher_model,
        messages=[{"role": "user", "content": make_prompt(row_tokens)}],
    )
    answer = response.choices[0].message.content

    predicted_tags = None
    format_error = None

    try:
        predicted_tags = parse_model_tags(answer, len(row_tokens), tag_names)
    except (ValueError, TypeError) as error:
        format_error = str(error)

    save_answer(pilot_path, {
        "row_index": row_index,
        "tokens": row_tokens,
        "model_id": teacher_model,
        "answer": answer,
        "predicted_tags": predicted_tags,
        "format_error": format_error,
    })

    print(f"Запись {row_index}: {format_error or 'метки сохранены'}")

print("Файл:", pilot_path)

Запись 0: метки сохранены
Запись 1: метки сохранены
Запись 2: метки сохранены
Запись 3: Число меток не совпадает с числом токенов
Запись 4: Число меток не совпадает с числом токенов
Файл: /content/drive/MyDrive/llm-ner-conll2003/runs/train_pilot_01.jsonl


Разметил первые пять записей из train, чтобы проверить, как сохраняются ответы GPT перед обучением. Для каждой записи сохранил её номер, исходные токены и ответ модели в поле answer. Если ответ прошёл проверку, записал список меток в predicted_tags. Если проверка не прошла, оставил там None и сохранил причину в format_error

In [15]:
pilot_pairs = []

with pilot_path.open(encoding="utf-8") as file:
    for line in file:
        saved = json.loads(line)
        row_index = saved["row_index"]
        record = train[row_index]

        if saved["tokens"] != record["tokens"]:
            raise ValueError(f"Не совпали токены в записи {row_index}")

        if saved["predicted_tags"] is None:
            print(f"Запись {row_index}: пока нет пригодных меток GPT")
            continue

        gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

        pilot_pairs.append({
            "row_index": row_index,
            "tokens": record["tokens"],
            "llm_tags": saved["predicted_tags"],
            "gold_tags": gold_tags,
        })

print("Записей для сравнения:", len(pilot_pairs))
print(pilot_pairs[0])

Запись 3: пока нет пригодных меток GPT
Запись 4: пока нет пригодных меток GPT
Записей для сравнения: 3
{'row_index': 0, 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'llm_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O'], 'gold_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']}


По номеру записи нашёл исходное предложение в train и проверил, что токены совпадают с сохранёнными. Для трёх пригодных ответов собрал метки GPT и эталонные метки того же предложения. Две записи с ошибкой длины пока пропустил, чтобы оба набора для обучения содержали одни и те же предложения

In [16]:
import random


def label_train_rows(row_indices, output_path):
    saved_indices = set()

    if output_path.exists():
        with output_path.open(encoding="utf-8") as file:
            for line in file:
                saved_indices.add(json.loads(line)["row_index"])

    for row_index in row_indices:
        if row_index in saved_indices:
            continue

        row_tokens = train[row_index]["tokens"]
        response = client.chat.completions.create(
            model=teacher_model,
            messages=[{"role": "user", "content": make_prompt(row_tokens)}],
        )
        answer = response.choices[0].message.content

        predicted_tags = None
        format_error = None
        try:
            predicted_tags = parse_model_tags(answer, len(row_tokens), tag_names)
        except (ValueError, TypeError) as error:
            format_error = str(error)

        save_answer(output_path, {
            "row_index": row_index,
            "tokens": row_tokens,
            "model_id": teacher_model,
            "answer": answer,
            "predicted_tags": predicted_tags,
            "format_error": format_error,
        })

        saved_indices.add(row_index)


audit_indices = random.Random(42).sample(range(5, len(train)), 50)
audit_path = runs_dir / "train_audit_01.jsonl"

label_train_rows(audit_indices, audit_path)
print("Ответы сохранены в", audit_path)

Ответы сохранены в /content/drive/MyDrive/llm-ner-conll2003/runs/train_audit_01.jsonl


label_train_rows - отправляет записи из train в GPT и сохраняет текст каждого ответа. Если список BIO-меток прошёл проверку длины и правил BIO, функция сохраняет этот список. При ошибке записывает её причину

Выбрал 50 случайных записей с seed 42, исключив первые пять, на которых проверял код. Сохранил ответы в train_audit_01.jsonl. По этому файлу проверю, какая доля ответов пригодна для обучения и насколько метки GPT совпадают с эталоном

Переношу функции из 1conll.ipynb, чтобы сравнить метки GPT с эталоном по тем же правилам, что на первом этапе: сущность засчитывается, если совпали обе границы и тип

In [17]:
def bio_to_spans(tags):
    spans = []
    i = 0

    while i < len(tags):
        tag = tags[i]
        if tag == "O":
            i += 1
            continue

        start = i
        entity_type = tag[2:]
        i += 1

        while i < len(tags) and tags[i] == f"I-{entity_type}":
            i += 1

        spans.append((start, i, entity_type))

    return spans

bio_to_spans - функция, которая собирает целые сущности из BIO-меток, чтобы потом сравнить их границы и типы

In [18]:
def count_matches(gold_tags, predicted_tags):
    if len(gold_tags) != len(predicted_tags):
        raise ValueError("Число меток в двух разметках должно совпадать.")

    gold = set(bio_to_spans(gold_tags))
    predicted = set(bio_to_spans(predicted_tags))

    tp = len(gold & predicted)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    return tp, fp, fn


def scores_from_counts(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return precision, recall, f1

В count_matches сравниваю ответ с эталоном и считаю, сколько сущностей нашли правильно (tp - true positive), сколько выделили лишних (fp - false positive) и сколько пропустили (fn - false negative)
В scores_from_counts считаю precision, recall и F1

In [21]:
with audit_path.open(encoding="utf-8") as file:
    audit_results = [json.loads(line) for line in file]

audit_tp, audit_fp, audit_fn = 0, 0, 0
failed_count = 0
failed_gold_count = 0

for result in audit_results:
    record = train[result["row_index"]]
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]
    predicted_tags = result["predicted_tags"]

    if predicted_tags is None:
        failed_count += 1
        failed_gold_count += len(bio_to_spans(gold_tags))
        continue

    tp, fp, fn = count_matches(gold_tags, predicted_tags)
    audit_tp += tp
    audit_fp += fp
    audit_fn += fn

_, _, valid_f1 = scores_from_counts(audit_tp, audit_fp, audit_fn)
_, _, full_f1 = scores_from_counts(
    audit_tp, audit_fp, audit_fn + failed_gold_count
)

print(f"Пригодных ответов: {len(audit_results) - failed_count}/{len(audit_results)}")
print(f"Micro-F1 на пригодных ответах: {valid_f1:.2%}")
print(f"Micro-F1 на всех 50 записях: {full_f1:.2%}")
print("Эталонных сущностей в записях без пригодного ответа:", failed_gold_count)

Пригодных ответов: 44/50
Micro-F1 на пригодных ответах: 80.67%
Micro-F1 на всех 50 записях: 68.57%
Эталонных сущностей в записях без пригодного ответа: 21


С первой попытки проверку прошли 44 ответа из 50. На них micro-F1 получился 80,67%. В шести записях без пригодных меток была 21 эталонная сущность. Если учесть их как пропуски, micro-F1 по всей выборке составляет 68,57%

Для шести записей, по которым первый ответ GPT не прошёл проверку, отправлю по одному повторному запросу с тем же промптом. Проверю, для скольких из них удастся получить пригодные метки для обучения

In [22]:
retry_indices = [
    result["row_index"]
    for result in audit_results
    if result["predicted_tags"] is None
]
retry_path = runs_dir / "train_audit_retry_01.jsonl"

label_train_rows(retry_indices, retry_path)

with retry_path.open(encoding="utf-8") as file:
    retry_results = [json.loads(line) for line in file]

recovered_count = sum(
    result["predicted_tags"] is not None
    for result in retry_results
)

print(f"После повтора получили пригодные метки: {recovered_count}/{len(retry_indices)}")

После повтора получили пригодные метки: 5/6


Повторил запрос для шести записей, по которым первый ответ не прошёл проверку. Пять новых ответов прошли проверку, поэтому теперь есть пригодные метки для 49 из 50 записей. Дальше сравню эти пять ответов с эталоном и пересчитаю micro-F1

In [23]:
retry_tp, retry_fp, retry_fn = audit_tp, audit_fp, audit_fn
remaining_failed_gold_count = failed_gold_count

for result in retry_results:
    predicted_tags = result["predicted_tags"]
    if predicted_tags is None:
        continue

    record = train[result["row_index"]]
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

    remaining_failed_gold_count -= len(bio_to_spans(gold_tags))
    tp, fp, fn = count_matches(gold_tags, predicted_tags)
    retry_tp += tp
    retry_fp += fp
    retry_fn += fn

_, _, retry_valid_f1 = scores_from_counts(retry_tp, retry_fp, retry_fn)
_, _, retry_full_f1 = scores_from_counts(
    retry_tp, retry_fp, retry_fn + remaining_failed_gold_count
)

print(f"Micro-F1 на пригодных ответах после повтора: {retry_valid_f1:.2%}")
print(f"Micro-F1 на всех 50 записях после повтора: {retry_full_f1:.2%}")

Micro-F1 на пригодных ответах после повтора: 81.25%
Micro-F1 на всех 50 записях после повтора: 80.25%


Для каждой из шести записей, где первый ответ GPT не прошёл проверку формата и правил BIO, отправил по одному повторному запросу с тем же промптом. При пересчёте метрик использовал первый ответ, если он прошёл проверку, и повторный, если проверку прошёл только он

На записях с пригодными ответами micro-F1 составил 81,25%. На всех 50 записях micro-F1 вырос с 68,57% до 80,25%. Эталонные сущности из записи, для которой обе попытки не прошли проверку, засчитал как пропущенные

При разметке train буду один раз повторять запрос для записей, где первый ответ не прошёл проверку формата и правил BIO